<a href="https://colab.research.google.com/github/mowumialabi/west-africa-climate-trap-2026/blob/main/era5_thermodynamic_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [6]:
import ee
import pandas as pd
import numpy as np

# =====================================================================
# CONFIGURATION: DEFINE YOUR GOOGLE CLOUD PROJECT ID HERE
# =====================================================================
# Replace this placeholder string with your actual Google Cloud/GEE Project ID
GEE_PROJECT_ID = 'ee-mowumialabi'

# =====================================================================
# STEP 1: EARTH ENGINE AUTHENTICATION & PROJECT INITIALIZATION
# =====================================================================
print("Initializing Google Earth Engine Connection...")
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"Earth Engine successfully initialized with project: {GEE_PROJECT_ID}")
except Exception as e:
    print("Authentication/initialization failed. Launching authorization flow...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"Earth Engine successfully authenticated and initialized with project: {GEE_PROJECT_ID}")

# =====================================================================
# STEP 2: DEFINE STUDY AREAS AND TIME FILTER
# =====================================================================
# Geographic coordinates capturing the West African latitudinal transect
sites = {
    "Ile-Ife": {"lat": 7.484, "lon": 4.484},   # Core Southwest Moisture Trap
    "Abuja":   {"lat": 9.076, "lon": 7.398},   # Mid-Belt Transition Front
    "Kano":    {"lat": 12.002, "lon": 8.592}   # Northern Arid Boundary
}

start_date = "2026-01-01"
end_date = "2026-02-01"

# =====================================================================
# STEP 3: INGEST & FILTER FOR PEAK AFTERNOON METEOROLOGICAL CONDITIONS
# =====================================================================
print(f"Ingesting peak afternoon data columns (12:00 - 16:00 UTC) from {start_date} to {end_date}...")

# Filter the collection by date and restrict to peak diurnal heating windows (12:00 to 16:00 UTC)
era5_hourly = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
               .filterDate(start_date, end_date)
               .filter(ee.Filter.calendarRange(12, 16, 'hour'))
               .select(["temperature_2m", "dewpoint_temperature_2m"]))

# Calculate the mean of these peak hours across the month
mean_climate_layer = era5_hourly.mean()

# =====================================================================
# STEP 4: SPATIAL REDUCTION (PIXEL EXTRACTION)
# =====================================================================
extracted_records = []

print("Extracting regional peak atmospheric variables...")
for name, coords in sites.items():
    point = ee.Geometry.Point([coords["lon"], coords["lat"]])

    # Reduce the region to sample pixel value at native 9km grid scale
    stats = mean_climate_layer.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=9000
    ).getInfo()

    # Thermodynamic Conversion: Kelvin to Celsius
    t_air_c = stats["temperature_2m"] - 273.15
    t_dew_c = stats["dewpoint_temperature_2m"] - 273.15

    extracted_records.append({
        "Site": name,
        "Latitude": f"{coords['lat']}°N",
        "Air_Temp_C": t_air_c,
        "Dew_Point_C": t_dew_c
    })

# Convert raw data matrix into a structured DataFrame
df_gee = pd.DataFrame(extracted_records)

# =====================================================================
# STEP 5: MATHEMATICAL CALCULATIONS (MAGNUS-TETENS FORMULATION)
# =====================================================================
def apply_magnus_tetens_physics(row):
    # Standard meteorological coefficients over water (T > 0 °C)
    c1 = 0.61078  # Saturation vapor pressure at 0°C (kPa)
    c2 = 17.27    # Dimensionless constant
    c3 = 237.3    # Temperature constant (°C)

    # 1. Saturation Vapor Pressure (es) at Air Temp
    es = c1 * np.exp((c2 * row["Air_Temp_C"]) / (c3 + row["Air_Temp_C"]))

    # 2. Actual Vapor Pressure (ea) at Dew Point Temp
    ea = c1 * np.exp((c2 * row["Dew_Point_C"]) / (c3 + row["Dew_Point_C"]))

    # 3. Derive Relative Humidity and Vapor Pressure Deficit (VPD)
    rh = (ea / es) * 100.0
    vpd = es - ea

    return pd.Series([es, ea, vpd, rh])

print("Applying thermodynamic physics equations...")
df_gee[["es_kPa", "ea_kPa", "VPD_kPa", "RH_Pct"]] = df_gee.apply(apply_magnus_tetens_physics, axis=1)

# =====================================================================
# STEP 6: OUTPUT GENERATION FOR MANUSCRIPT DATA MATRIX
# =====================================================================
print("\n" + "="*65)
print("=== VALIDATED PEAK DIURNAL GEOSPATIAL EXTRACTION TABLE ===")
print("="*65)
print(df_gee.round({
    "Air_Temp_C": 1, "Dew_Point_C": 1,
    "es_kPa": 4, "ea_kPa": 4, "VPD_kPa": 4, "RH_Pct": 1
}).to_string(index=False))


Initializing Google Earth Engine Connection...
Earth Engine successfully initialized with project: ee-mowumialabi
Ingesting peak afternoon data columns (12:00 - 16:00 UTC) from 2026-01-01 to 2026-02-01...
Extracting regional peak atmospheric variables...
Applying thermodynamic physics equations...

=== VALIDATED PEAK DIURNAL GEOSPATIAL EXTRACTION TABLE ===
   Site Latitude  Air_Temp_C  Dew_Point_C  es_kPa  ea_kPa  VPD_kPa  RH_Pct
Ile-Ife  7.484°N        33.7         16.4  5.2374  1.8706   3.3668    35.7
  Abuja  9.076°N        35.1          7.0  5.6599  0.9993   4.6606    17.7
   Kano 12.002°N        31.9         -1.0  4.7205  0.5688   4.1517    12.0


In [ ]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [ ]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.0-flash', stream=True):
  wrapper.print(chunk)

Alright, settle in, because the Roman Empire’s evolution wasn't a tidy, linear
process. It was a centuries-long, tumultuous transformation, marked by
breathtaking innovation, brutal power struggles, and a slow, creeping societal
decay. We're talking about a journey from a humble city-state in the Italian
peninsula to a sprawling, multifaceted empire that left an indelible mark on
law, language, architecture, governance, and even our very understanding of the
world.

It all began, as legend would have it, with Romulus and Remus, twin brothers
raised by a she-wolf, who founded the city of Rome in 753 BCE. Now, that’s just
a legend, but it serves to highlight the foundational spirit of Rome: ambition,
strength, and a certain ruthlessness. Initially, Rome was ruled by a monarchy, a
system eventually deemed unsatisfactory by the powerful patrician class. This
led to the **Roman Republic**, established around 509 BCE, a watershed moment
that would define the early character of Rome.

The Rep